In [ ]:
%matplotlib widget
%reset -f
%load_ext autoreload
%autoreload 1
%aimport mechanics

from sympy import *
from mechanics import *

t, = base_spaces('t')
def dot(f): return diff(f, t)

q = theta1, theta2 = variables(r'\theta_1 \theta_2', t, space=space.S)
dq = tuple(dot(q_n) for q_n in q)
ddq = tuple(dot(dq_n) for dq_n in dq)

g, m1, m2, l1, l2 = constants(r'g, m_1 m_2 \ell_1 \ell_2')

x1 = l1 * sin(theta1)
y1 = -l1 * cos(theta1)
x2 = x1 + l2 * sin(theta2)
y2 = y1 - l2 * cos(theta2)

U = m1 * g * y1 + m2 * g * y2
T = (m1 / 2 * (dot(x1)**2 + dot(y1)**2)
     + m2 / 2 * (dot(x2)**2 + dot(y2)**2)).simplify()
E = T + U
L = T - U
show('L =', L)

from mechanics.lagrange import euler_lagrange_equation

EL = euler_lagrange_equation(L, q)
show_equations(EL)

F = solve(EL, ddq)
show_equations(F)


In [ ]:
%autoreload

from mechanics.difference import forward
from mechanics.space import Z
from mechanics.lagrange import discrete_euler_lagrange_equation

i, = indices('i')
h, T = constants('h T')
N, = constants('N', space=Z)

d = discretization().space(t, i, h).diff(t, forward)
theta1, theta2 = d(q)

L_trapezoidal = (
    L.subs(zip(dq, forward(d(q), i, h))).subs(zip(q, d(q)))
    + L.subs(zip(dq, forward(d(q), i, h))).subs(zip(q, shift_index(d(q), i, 1)))
) / 2 * h
L_trapezoidal = L_trapezoidal.simplify()
show('L_d =', L_trapezoidal)

L_midpoint = L.subs(zip(dq, forward(d(q), i, h))).subs(zip(q, [(d(q_n) + shift_index(d(q_n), i, 1)) / 2 for q_n in q])) * h
L_midpoint = L_midpoint.simplify()
show('L_d =', L_midpoint)

EL = discrete_euler_lagrange_equation(L_trapezoidal, d(q))
EL = discrete_euler_lagrange_equation(L_midpoint, d(q))
show_equations(EL)

In [ ]:
%autoreload
solver = build_solver()
solver.constants(g, m1, m2, l1, l2)
solver.constants(h, T)
solver.variables(*d(q), index=(i, -1, T/h))
solver.inputs(*[q[i] for q in d(q) for i in [0, -1]])
solver.functions('x1', 'y1', 'x2', 'y2', 'E', index=(i, 0, T/h))
with solver.steps(i, 0, T/h) as step:
    with step.newton() as newton:
        newton.unknowns(*[q[i+1] for q in d(q)])
        newton.initial_guess({q[i+1]: q[i] for q in d(q)})
        newton.implicit(EL)
    step.calculate({'x1': d(x1), 'y1': d(y1), 'x2': d(x2), 'y2': d(y2), 'E': d(E)}, i)
solver = solver.generate()

In [ ]:
%autoreload
import numpy as np
_ = solver.run({
    m1: 1.0, m2: 1.0, l1: 1.0, l2: 1.0, g: 1.0,
    h: 0.1, T: 100.0,
    theta1[-1]: np.pi/2, theta2[-1]: 0.0,
    theta1[0]: np.pi/2, theta2[0]: 0.0,
})

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

t = np.arange(0, _[T], _[h])

fig, ax = plt.subplots(1, 1, figsize=(4, 4), tight_layout=True)
ax.plot(_['x1'], _['y1'], label='$m_1$')
ax.plot(_['x2'], _['y2'], label='$m_2$')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_aspect('equal')
ax.legend()

fig, axes = plt.subplots(3, 1, figsize=(6, 8), tight_layout=True)
variables = [(r'\theta', (theta1, theta2)),
            #  (r'v_\theta', (v_theta1, v_theta2)), 
             ('E', ('E',))]
for ax, (name, vars) in zip(axes.flatten(), variables):
    for var in vars:
        ax.plot(t, _[var][1:len(t)+1], label=f'${getattr(var, "name", var)}$')
    ax.set_xlabel('$t$')
    ax.set_ylabel(f'${name}$')
    ax.set_xlim(0, _[T])
    if len(vars) > 1:
        ax.legend()